# RAG Module — Hieroglyph Transliteration → German (Full Dataset v2)

## Multi-Strategy Retrieval-Augmented Generation

## 1. Install Dependencies

In [1]:
import subprocess, sys

def _pip(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

for pkg in ['scikit-learn', 'rank-bm25', 'python-Levenshtein', 'pandas', 'numpy']:
    try:
        if pkg == 'scikit-learn':
            import sklearn
        elif pkg == 'rank-bm25':
            from rank_bm25 import BM25Okapi
        elif pkg == 'python-Levenshtein':
            import Levenshtein
        elif pkg == 'pandas':
            import pandas
        elif pkg == 'numpy':
            import numpy
    except ImportError:
        _pip(pkg)

print("Dependencies ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 62.8 MB/s eta 0:00:00
Dependencies ready.


## 2. Imports

In [2]:
import re
import unicodedata
import pickle
from pathlib import Path
from typing import List, Dict, Optional, Tuple, Any, Set
from dataclasses import dataclass
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

try:
    import Levenshtein
except ImportError:
    Levenshtein = None

print("Imports done.")

Imports done.


## 3. Text Normalization

In [3]:
import unicodedata

APOSTROPHES = {
    '\u02be': "'", '\u2018': "'", '\u2019': "'", '\u201b': "'",
    '\u2032': "'", '`': "'", '\u00b4': "'", '\u02bf': "'",
}

_NONE_STRINGS = {'', 'None', 'nan', 'NaN', 'none', 'null', 'NULL'}


def normalize_text(text: str) -> str:
    if text is None:
        return ''
    text = str(text).strip()
    if text in _NONE_STRINGS:
        return ''
    text = unicodedata.normalize('NFC', text)
    for src_ch, tgt_ch in APOSTROPHES.items():
        text = text.replace(src_ch, tgt_ch)
    text = text.replace('\u0ea4', '\ua723')
    text = text.replace('\u02bf', '\ua725')
    text = re.sub(r'[.()<>\[\],;|\-_=/\\]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def tokenize_words(text: str) -> List[str]:
    return normalize_text(text).split()


def char_sort_token(token: str) -> str:
    return ''.join(sorted(token))


def char_sorted_key(text: str) -> str:
    words = normalize_text(text).split()
    sorted_tokens = sorted(char_sort_token(w) for w in words)
    return ' '.join(sorted_tokens)

print("Normalization + char-sort sanity check:")
tests = [
    ('htp di mnr', 'di mnr htp',  True,  'word swap'),
    ('htp di mnr', 'htp di mrn',  True,  'char scramble'),
    ('htp di mnr', 'di mrn htp',  True,  'both swap + scramble'),
    ('htp di mnr', 'htp di xyz',  False, 'different token'),
]
for a, b, expect, desc in tests:
    ka, kb = char_sorted_key(a), char_sorted_key(b)
    match = ka == kb
    ok = '✅' if match == expect else '❌'
    print(f'  {ok} {desc:25s}: "{a}" vs "{b}" => {ka} vs {kb} => {"MATCH" if match else "DIFF"}')

Normalization + char-sort sanity check:
  ✅ word swap                : "htp di mnr" vs "di mnr htp" => di hpt mnr vs di hpt mnr => MATCH
  ✅ char scramble            : "htp di mnr" vs "htp di mrn" => di hpt mnr vs di hpt mnr => MATCH
  ✅ both swap + scramble     : "htp di mnr" vs "di mrn htp" => di hpt mnr vs di hpt mnr => MATCH
  ✅ different token          : "htp di mnr" vs "htp di xyz" => di hpt mnr vs di hpt xyz => DIFF


## 4. Similarity Metrics

In [4]:
def jaccard_similarity(words_a: List[str], words_b: List[str]) -> float:
    set_a, set_b = set(words_a), set(words_b)
    if not set_a and not set_b:
        return 1.0
    inter = len(set_a & set_b)
    union = len(set_a | set_b)
    return inter / union if union else 0.0


def dice_coefficient(words_a: List[str], words_b: List[str]) -> float:
    set_a, set_b = set(words_a), set(words_b)
    if not set_a and not set_b:
        return 1.0
    inter = len(set_a & set_b)
    total = len(set_a) + len(set_b)
    return (2 * inter) / total if total else 0.0


def overlap_coefficient(words_a: List[str], words_b: List[str]) -> float:
    set_a, set_b = set(words_a), set(words_b)
    if not set_a or not set_b:
        return 0.0
    inter = len(set_a & set_b)
    return inter / min(len(set_a), len(set_b))


def sorted_levenshtein_ratio(words_a: List[str], words_b: List[str]) -> float:
    s_a = ' '.join(sorted(words_a))
    s_b = ' '.join(sorted(words_b))
    if not s_a and not s_b:
        return 1.0
    if Levenshtein is not None:
        return float(Levenshtein.ratio(s_a, s_b))
    max_len = max(len(s_a), len(s_b), 1)
    edits = _edit_distance(s_a, s_b)
    return 1.0 - edits / max_len


def sequence_levenshtein_ratio(words_a: List[str], words_b: List[str]) -> float:
    s_a = ' '.join(words_a)
    s_b = ' '.join(words_b)
    if not s_a and not s_b:
        return 1.0
    if s_a == s_b:
        return 1.0
    if Levenshtein is not None:
        return float(Levenshtein.ratio(s_a, s_b))
    max_len = max(len(s_a), len(s_b), 1)
    edits = _edit_distance(s_a, s_b)
    return 1.0 - edits / max_len


def _edit_distance(a: str, b: str) -> int:
    if len(a) < len(b):
        a, b = b, a
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(cur[-1] + 1, prev[j] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]


print("Similarity metrics ready.")

Similarity metrics ready.


## 5. The HieroglyphRAG Class

In [5]:
@dataclass
class RAGResult:
    matched_src: Optional[str]
    matched_tgt: Optional[str]
    similarity: float
    confidence: str
    method: str
    metrics: Dict[str, float]
    top_k: List[Dict[str, Any]]
    all_targets: Optional[List[str]] = None


def _whitespace_tokenizer(s: str) -> List[str]:
    return s.split()


class HieroglyphRAG:
    def __init__(
        self,
        high_threshold: float = 0.85,
        medium_threshold: float = 0.45,
        top_k: int = 100,
        weights: Optional[Dict[str, float]] = None,
    ):
        self.high_threshold = high_threshold
        self.medium_threshold = medium_threshold
        self.top_k = top_k
        self.weights = weights or {
            'jaccard':     0.15,
            'dice':        0.10,
            'overlap':     0.10,
            'sorted_lev':  0.20,
            'seq_lev':     0.25,
            'tfidf_cos':   0.20,
        }
        self.df = None
        self.src_col = None
        self.tgt_col = None
        self.normalized_sources: List[str] = []
        self.tokenized_sources: List[List[str]] = []
        self.targets: List[str] = []
        
        self.exact_string_index: Dict[str, List[int]] = {}
        self.exact_wordset_index: Dict[str, List[int]] = {}
        self.char_sorted_index: Dict[str, List[int]] = {}   # NEW
        
        self.src_to_all_targets: Dict[str, Set[str]] = {}
        
        self.tfidf: Optional[TfidfVectorizer] = None
        self.tfidf_matrix = None
        self.char_tfidf: Optional[TfidfVectorizer] = None
        self.char_tfidf_matrix = None
        self.bm25: Optional[BM25Okapi] = None

    def build(self, df: pd.DataFrame, src_col: str, tgt_col: str) -> 'HieroglyphRAG':
        self.df = df.reset_index(drop=True).copy()
        self.src_col = src_col
        self.tgt_col = tgt_col

        self.normalized_sources = [normalize_text(s) for s in self.df[src_col]]
        self.tokenized_sources = [s.split() for s in self.normalized_sources]
        self.targets = self.df[tgt_col].astype(str).tolist()

        self.src_to_all_targets = {}
        for idx, norm in enumerate(self.normalized_sources):
            if norm:
                self.src_to_all_targets.setdefault(norm, set()).add(self.targets[idx].strip())

        self.exact_string_index = {}
        for idx, norm in enumerate(self.normalized_sources):
            if norm:
                self.exact_string_index.setdefault(norm, []).append(idx)

        self.exact_wordset_index = {}
        for idx, words in enumerate(self.tokenized_sources):
            key = ' '.join(sorted(set(words)))
            if key:
                self.exact_wordset_index.setdefault(key, []).append(idx)

        self.char_sorted_index = {}
        for idx, words in enumerate(self.tokenized_sources):
            sorted_tokens = sorted(char_sort_token(w) for w in words)
            key = ' '.join(sorted_tokens)
            if key:
                self.char_sorted_index.setdefault(key, []).append(idx)

        self.tfidf = TfidfVectorizer(
            tokenizer=_whitespace_tokenizer,
            lowercase=False,
            token_pattern=None,
        )
        self.tfidf_matrix = self.tfidf.fit_transform(self.normalized_sources)

        self.char_tfidf = TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(2, 4),
            lowercase=False,
        )
        self.char_tfidf_matrix = self.char_tfidf.fit_transform(self.normalized_sources)

        self.bm25 = BM25Okapi(self.tokenized_sources)

        n_dup_str = sum(1 for v in self.exact_string_index.values() if len(v) > 1)
        n_multi_tgt = sum(1 for v in self.src_to_all_targets.values() if len(v) > 1)
        print(f'RAG index built: {len(self.df):,} entries')
        print(f'  Exact string keys      : {len(self.exact_string_index):,} unique')
        print(f'  Word-set keys          : {len(self.exact_wordset_index):,} unique')
        print(f'  Char-sorted keys       : {len(self.char_sorted_index):,} unique')
        print(f'  Multi-translation srcs : {n_multi_tgt:,} (same src, different German)')
        print(f'  TF-IDF vocab           : {len(self.tfidf.vocabulary_):,} words')
        print(f'  Char n-gram vocab      : {len(self.char_tfidf.vocabulary_):,} ngrams')
        return self

    def _best_from_indices(self, indices: List[int]) -> int:
        if len(indices) == 1:
            return indices[0]
        return max(indices, key=lambda i: len(self.targets[i]))

    def _all_targets_from_indices(self, indices: List[int]) -> List[str]:
        return list(set(self.targets[i].strip() for i in indices))

    def lookup(self, query: str) -> RAGResult:
        norm = normalize_text(query)
        tokens = norm.split()

        if not tokens:
            return RAGResult(
                matched_src=None, matched_tgt=None,
                similarity=0.0, confidence='LOW',
                method='empty_query', metrics={}, top_k=[],
                all_targets=[],
            )

        if norm in self.exact_string_index:
            indices = self.exact_string_index[norm]
            idx = self._best_from_indices(indices)
            all_tgts = self._all_targets_from_indices(indices)
            return RAGResult(
                matched_src=self.df.iloc[idx][self.src_col],
                matched_tgt=self.targets[idx],
                similarity=1.0,
                confidence='HIGH',
                method='exact_string',
                metrics={'exact_string': 1.0},
                top_k=[{'idx': int(idx), 'src': self.df.iloc[idx][self.src_col],
                         'tgt': self.targets[idx], 'combined': 1.0}],
                all_targets=all_tgts,
            )

        exact_key = ' '.join(sorted(set(tokens)))
        if exact_key in self.exact_wordset_index:
            indices = self.exact_wordset_index[exact_key]
            idx = self._best_from_indices(indices)
            all_tgts = self._all_targets_from_indices(indices)
            return RAGResult(
                matched_src=self.df.iloc[idx][self.src_col],
                matched_tgt=self.targets[idx],
                similarity=1.0,
                confidence='HIGH',
                method='exact_word_set',
                metrics={'exact_word_set': 1.0},
                top_k=[{'idx': int(idx), 'src': self.df.iloc[idx][self.src_col], 'tgt': self.targets[idx], 'combined': 1.0}],
                all_targets=all_tgts,
            )

        cs_tokens = sorted(char_sort_token(w) for w in tokens)
        cs_key = ' '.join(cs_tokens)
        if cs_key in self.char_sorted_index:
            indices = self.char_sorted_index[cs_key]
            idx = self._best_from_indices(indices)
            all_tgts = self._all_targets_from_indices(indices)
            return RAGResult(
                matched_src=self.df.iloc[idx][self.src_col],
                matched_tgt=self.targets[idx],
                similarity=1.0,
                confidence='HIGH',
                method='char_sorted_match',
                metrics={'char_sorted_match': 1.0},
                top_k=[{'idx': int(idx), 'src': self.df.iloc[idx][self.src_col],
                         'tgt': self.targets[idx], 'combined': 1.0}],
                all_targets=all_tgts,
            )

        query_vec = self.tfidf.transform([norm])
        cos_scores = cosine_similarity(query_vec, self.tfidf_matrix).ravel()

        char_query_vec = self.char_tfidf.transform([norm])
        char_cos_scores = cosine_similarity(char_query_vec, self.char_tfidf_matrix).ravel()

        bm25_scores = self.bm25.get_scores(tokens)
        bm25_max = bm25_scores.max() if bm25_scores.max() > 0 else 1.0
        bm25_norm = bm25_scores / bm25_max

        combined_retrieval = 0.35 * cos_scores + 0.30 * bm25_norm + 0.35 * char_cos_scores
        top_idx = np.argsort(combined_retrieval)[::-1][:self.top_k]

        best_idx = -1
        best_score = -1.0
        best_metrics: Dict[str, float] = {}
        top_k_info: List[Dict[str, Any]] = []

        for idx in top_idx:
            cand_tokens = self.tokenized_sources[idx]
            jac = jaccard_similarity(tokens, cand_tokens)
            dic = dice_coefficient(tokens, cand_tokens)
            ovl = overlap_coefficient(tokens, cand_tokens)
            slev = sorted_levenshtein_ratio(tokens, cand_tokens)
            seq_lev = sequence_levenshtein_ratio(tokens, cand_tokens)
            tfidf_cos = float(cos_scores[idx])

            score = (
                self.weights['jaccard']    * jac +
                self.weights['dice']       * dic +
                self.weights['overlap']    * ovl +
                self.weights['sorted_lev'] * slev +
                self.weights['seq_lev']    * seq_lev +
                self.weights['tfidf_cos']  * tfidf_cos
            )

            metrics = {
                'jaccard': float(jac), 'dice': float(dic),
                'overlap': float(ovl), 'sorted_lev': float(slev),
                'seq_lev': float(seq_lev), 'tfidf_cosine': tfidf_cos,
                'char_cosine': float(char_cos_scores[idx]),
                'bm25': float(bm25_scores[idx]),
            }

            top_k_info.append({
                'idx': int(idx), 'src': self.df.iloc[idx][self.src_col],
                'tgt': self.targets[idx], 'combined': float(score), **metrics,
            })

            if score > best_score:
                best_score = score
                best_idx = int(idx)
                best_metrics = metrics

        top_k_info.sort(key=lambda x: x['combined'], reverse=True)

        if best_score >= self.high_threshold:
            confidence, method = 'HIGH', 'rerank_high'
        elif best_score >= self.medium_threshold:
            confidence, method = 'MEDIUM', 'rerank_medium'
        else:
            confidence, method = 'LOW', 'rerank_low'

        return RAGResult(
            matched_src=self.df.iloc[best_idx][self.src_col] if best_idx >= 0 else None,
            matched_tgt=self.targets[best_idx] if best_idx >= 0 else None,
            similarity=float(best_score),
            confidence=confidence,
            method=method,
            metrics=best_metrics,
            top_k=top_k_info[:10],
            all_targets=None,
        )

    def translate(self, query: str, fallback_fn=None) -> Tuple[str, Dict[str, Any]]:
        result = self.lookup(query)
        info = {
            'source': 'RAG',
            'confidence': result.confidence,
            'similarity': result.similarity,
            'method': result.method,
            'metrics': result.metrics,
            'matched_src': result.matched_src,
            'all_targets': result.all_targets,
        }

        if result.confidence in ('HIGH', 'MEDIUM'):
            return result.matched_tgt, info

        if fallback_fn is not None:
            info['source'] = 'FALLBACK_MODEL'
            info['rag_best_similarity'] = result.similarity
            info['rag_best_match'] = result.matched_tgt
            translation = fallback_fn(query)
            return translation, info

        return result.matched_tgt or '', info

    def save(self, path: str) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, 'wb') as f:
            pickle.dump(self, f)
        print(f'RAG saved -> {path}')

    @staticmethod
    def load(path: str) -> 'HieroglyphRAG':
        with open(path, 'rb') as f:
            return pickle.load(f)


print("HieroglyphRAG class defined (5-level matching).")

HieroglyphRAG class defined (5-level matching).


## 6. Build RAG on ALL Data


In [6]:
import os

RAG_DATA_PATH = '/kaggle/input/datasets/mo3azkhaled/nlp-dataset/dataset_cleaned.csv'
candidates = [
    RAG_DATA_PATH,
    '/kaggle/input/datasets/moaztest105/newdaaaata/dataset_cleaned.csv',
    './dataset_cleaned.csv',
]
for c in candidates:
    if os.path.exists(c):
        RAG_DATA_PATH = c
        break

SRC_COL = 'clean_transliteration'
TGT_COL = 'clean_german'

print(f'Loading data: {RAG_DATA_PATH}')
df_full = pd.read_csv(RAG_DATA_PATH).dropna(subset=[SRC_COL, TGT_COL])
df_full[SRC_COL] = df_full[SRC_COL].astype(str).str.strip()
df_full[TGT_COL] = df_full[TGT_COL].astype(str).str.strip()

df_full = df_full[~df_full[SRC_COL].isin(['None', 'nan', 'NaN', '', 'null'])]
df_full = df_full[~df_full[TGT_COL].isin(['None', 'nan', 'NaN', '', 'null'])]
df_full = df_full[(df_full[SRC_COL].str.len() > 0) & (df_full[TGT_COL].str.len() > 0)]
df_full = df_full[df_full[SRC_COL].str.split().str.len().between(1, 80)]
df_full = df_full[df_full[TGT_COL].str.split().str.len().between(1, 80)]
df_full = df_full.drop_duplicates(subset=[SRC_COL, TGT_COL]).reset_index(drop=True)

print(f'Total rows: {len(df_full):,}')

rag = HieroglyphRAG(
    high_threshold=0.85,
    medium_threshold=0.65,
    top_k=100,
)
rag.build(df_full, src_col=SRC_COL, tgt_col=TGT_COL)

SAVE_PATH = '/kaggle/working/rag_index.pkl'
if not os.path.exists('/kaggle/working'):
    SAVE_PATH = './rag_index.pkl'
rag.save(SAVE_PATH)

Loading data: /kaggle/input/datasets/mo3azkhaled/nlp-dataset/dataset_cleaned.csv
Total rows: 89,562
RAG index built: 89,562 entries
  Exact string keys      : 84,908 unique
  Word-set keys          : 84,130 unique
  Char-sorted keys       : 84,458 unique
  Multi-translation srcs : 3,201 (same src, different German)
  TF-IDF vocab           : 21,346 words
  Char n-gram vocab      : 41,974 ngrams
RAG saved -> /kaggle/working/rag_index.pkl


## 7. Test the RAG (all 5 matching levels)

In [7]:
sample_idx = 100
sample_src = df_full.iloc[sample_idx][SRC_COL]
sample_tgt = df_full.iloc[sample_idx][TGT_COL]

print(f'Sample: "{sample_src}" -> "{sample_tgt}"')
print()

print('=' * 60)
print('TEST 1 — Exact string match')
print('=' * 60)
r = rag.lookup(sample_src)
print(f'  Method: {r.method}  Sim: {r.similarity}  Confidence: {r.confidence}')
print(f'  Result: {r.matched_tgt}')
if r.all_targets:
    print(f'  All valid targets ({len(r.all_targets)}): {r.all_targets[:3]}')
assert r.method == 'exact_string'
print('PASSED')

print()
print('=' * 60)
print('TEST 2 — Word swap (htp di mnr -> di mnr htp)')
print('=' * 60)
import random
words = sample_src.split()
if len(words) > 1:
    random.Random(42).shuffle(words)
    reordered = ' '.join(words)
    print(f'  Original : {sample_src}')
    print(f'  Reordered: {reordered}')
    r = rag.lookup(reordered)
    print(f'  Method: {r.method}  Sim: {r.similarity}  Confidence: {r.confidence}')
    assert r.confidence == 'HIGH'
    print('PASSED')

print()
print('=' * 60)
print('TEST 3 — Char scramble within token')
print('=' * 60)
words = sample_src.split()
if len(words) >= 1:
    last_word = list(words[-1])
    if len(last_word) > 1:
        last_word[0], last_word[-1] = last_word[-1], last_word[0]
    scrambled_words = words[:-1] + [''.join(last_word)]
    scrambled = ' '.join(scrambled_words)
    print(f'  Original : {sample_src}')
    print(f'  Scrambled: {scrambled}')
    r = rag.lookup(scrambled)
    print(f'  Method: {r.method}  Sim: {r.similarity}  Confidence: {r.confidence}')
    if r.method == 'char_sorted_match':
        print('Char-sorted match worked!')
    elif r.method in ('exact_string', 'exact_word_set'):
        print('Hit exact match (scrambled = same string by coincidence)')
    else:
        print(f'Fell through to {r.method} (token may have matched another entry)')

print()
print('=' * 60)
print('TEST 4 — Word swap + char scramble combined')
print('=' * 60)
if len(words) > 1 and len(words[-1]) > 1:
    random.Random(99).shuffle(scrambled_words)
    combined = ' '.join(scrambled_words)
    print(f'  Original: {sample_src}')
    print(f'  Combined: {combined}')
    r = rag.lookup(combined)
    print(f'  Method: {r.method}  Sim: {r.similarity}  Confidence: {r.confidence}')
    print(f'  Confidence: {r.confidence}')

print()
print('=' * 60)
print('TEST 5 — Random garbage')
print('=' * 60)
r = rag.lookup('xyz abcd unknown garbage')
print(f'  Method: {r.method}  Sim: {r.similarity}  Confidence: {r.confidence}')
assert r.confidence == 'LOW'
print('PASSED')

Sample: "wsrw nꞽt m n k ꞽrt ḥrw ꞽtḥtn f" -> "Osiris Neith, nimm dir das Horusauge, das er herausgezogen hat."

TEST 1 — Exact string match
  Method: exact_string  Sim: 1.0  Confidence: HIGH
  Result: Osiris Neith, nimm dir das Horusauge, das er herausgezogen hat.
  All valid targets (1): ['Osiris Neith, nimm dir das Horusauge, das er herausgezogen hat.']
PASSED

TEST 2 — Word swap (htp di mnr -> di mnr htp)
  Original : wsrw nꞽt m n k ꞽrt ḥrw ꞽtḥtn f
  Reordered: n ḥrw ꞽtḥtn k f m ꞽrt wsrw nꞽt
  Method: exact_word_set  Sim: 1.0  Confidence: HIGH
PASSED

TEST 3 — Char scramble within token
  Original : wsrw nꞽt m n k ꞽrt ḥrw ꞽtḥtn f
  Scrambled: wsrw nꞽt m n k ꞽrt ḥrw ꞽtḥtn f
  Method: exact_string  Sim: 1.0  Confidence: HIGH
Hit exact match (scrambled = same string by coincidence)

TEST 4 — Word swap + char scramble combined

TEST 5 — Random garbage
  Method: rerank_low  Sim: 0.18717948717948718  Confidence: LOW
PASSED


## 8. Top-K Inspection

In [8]:
query = sample_src
r = rag.lookup(query)
print(f'Query: {query}')
print(f'Best: sim={r.similarity:.4f} ({r.confidence}) method={r.method}')
if r.all_targets:
    print(f'All valid targets: {r.all_targets[:5]}')
print()
print('Top-10 candidates:')
print('-' * 110)
print(f'{"rank":<5}{"combined":<10}{"jaccard":<10}{"dice":<10}{"overlap":<10}{"sorted_lev":<12}{"cosine":<10}{"bm25":<10}{"src":<35}')
print('-' * 110)
for rank, c in enumerate(r.top_k, 1):
    src_preview = c['src'][:32] + '...' if len(c['src']) > 35 else c['src']
    bm25_val = c.get('bm25', 0.0)
    bm25_str = f"{'inf':<10}" if bm25_val == float('inf') else f"{bm25_val:<10.2f}"
    print(
        f"{rank:<5}{c['combined']:<10.4f}{c.get('jaccard', 0.0):<10.4f}{c.get('dice', 0.0):<10.4f}"
        f"{c.get('overlap', 0.0):<10.4f}{c.get('sorted_lev', 0.0):<12.4f}"
        f"{c.get('tfidf_cosine', 0.0):<10.4f}{bm25_str}{src_preview:<35}"
    )
print('-' * 110)

Query: wsrw nꞽt m n k ꞽrt ḥrw ꞽtḥtn f
Best: sim=1.0000 (HIGH) method=exact_string
All valid targets: ['Osiris Neith, nimm dir das Horusauge, das er herausgezogen hat.']

Top-10 candidates:
--------------------------------------------------------------------------------------------------------------
rank combined  jaccard   dice      overlap   sorted_lev  cosine    bm25      src                                
--------------------------------------------------------------------------------------------------------------
1    1.0000    0.0000    0.0000    0.0000    0.0000      0.0000    0.00      wsrw nꞽt m n k ꞽrt ḥrw ꞽtḥtn f     
--------------------------------------------------------------------------------------------------------------


## 9. Qualitative Results — 50 Samples

In [9]:
N_QUAL = 50
df_qual = df_full.sample(n=min(N_QUAL, len(df_full)), random_state=42).reset_index(drop=True)

rows = []
for _, row in df_qual.iterrows():
    src  = row[SRC_COL]
    exp  = row[TGT_COL]
    res  = rag.lookup(src)
    pred = res.matched_tgt if res.matched_tgt else ''
    
    strict_match = int(exp.strip() == pred.strip())
    
    all_tgts = res.all_targets or []
    any_valid_match = int(pred.strip() in [t.strip() for t in all_tgts]) if all_tgts else strict_match
    
    rows.append({
        'clean_transliteration': src,
        'expected': exp,
        'rag_prediction': pred,
        'similarity': round(res.similarity, 4),
        'confidence': res.confidence,
        'method': res.method,
        'exact_match': strict_match,
        'any_valid_match': any_valid_match,
        'n_translations': len(all_tgts),
    })

df_display = pd.DataFrame(rows)

n_strict = df_display['exact_match'].sum()
n_any    = df_display['any_valid_match'].sum()
n_high   = (df_display['confidence'] == 'HIGH').sum()

print(f'Strict exact match  : {n_strict}/{N_QUAL}  ({100*n_strict/N_QUAL:.1f}%)')
print(f'Any-valid match     : {n_any}/{N_QUAL}  ({100*n_any/N_QUAL:.1f}%)')
print(f'HIGH confidence     : {n_high}/{N_QUAL}  ({100*n_high/N_QUAL:.1f}%)')
print()
print('Method breakdown:')
for method, count in df_display['method'].value_counts().items():
    print(f'  {method:25s} : {count}')
print()

pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_rows', 60)
display(df_display[['clean_transliteration', 'expected', 'rag_prediction', 'similarity', 'confidence', 'method', 'exact_match', 'any_valid_match', 'n_translations']])

Strict exact match  : 50/50  (100.0%)
Any-valid match     : 50/50  (100.0%)
HIGH confidence     : 50/50  (100.0%)

Method breakdown:
  exact_string              : 50



,clean_transliteration,expected,rag_prediction,similarity,confidence,method,exact_match,any_valid_match,n_translations
0,jt f Jmn r ḥꜣt f,"sein Vater, Amunerhatef","sein Vater, Amunerhatef",1.0,HIGH,exact_string,1,1,1
1,jw j r ḏi̯t mnw k n Jwnw,Ich werde dich dauern lassen in Heliopolis.,Ich werde dich dauern lassen in Heliopolis.,1.0,HIGH,exact_string,1,1,1
2,m dj ꜣtj w,Laß sie nicht elend sein!,Laß sie nicht elend sein!,1.0,HIGH,exact_string,1,1,1
3,Wsjr ḫntj jmntt jmy m ḥwt nṯr,"Osiris-Chontamenti, komm in den Tempel!","Osiris-Chontamenti, komm in den Tempel!",1.0,HIGH,exact_string,1,1,1
4,ꞽw f wꜣḫi̯ f,Es geht ihm gut ist gesättigt erholt sich:,Es geht ihm gut ist gesättigt erholt sich:,1.0,HIGH,exact_string,1,1,1
5,ḏi̯ ṯ n j wꜣt nfrt swꜣ j,"Mögest du mir einen guten Weg geben, damit ich...","Mögest du mir einen guten Weg geben, damit ich...",1.0,HIGH,exact_string,1,1,1
6,rs tꜣꞽt m ḥtpw,Möge Tait in Frieden erwachen!,Möge Tait in Frieden erwachen!,1.0,HIGH,exact_string,1,1,2
7,zꜣt f Ḥpw,His daughter Hepu.,His daughter Hepu.,1.0,HIGH,exact_string,1,1,1
8,ḥꜣtj sn r ṯꜣi̯t nꜣy sn,"den Mut, um ihre. zu ergreifen.","den Mut, um ihre. zu ergreifen.",1.0,HIGH,exact_string,1,1,1
9,zꜣt wꜥb Jmn Nn nḫnt,"Die Tochter des Wab-Priesters des Amun, Nen-di...","Die Tochter des Wab-Priesters des Amun, Nen-di...",1.0,HIGH,exact_string,1,1,1


## 10. Full Evaluation

**Two exact-match metrics:**
- `strict_exact` — prediction == this specific expected translation
- `any_valid_exact` — prediction matches ANY known translation for this source

The `any_valid_exact` is the fair metric since the same transliteration can have multiple valid German translations.


In [ ]:
import math

def _ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

def bleu_n(reference: str, hypothesis: str, n: int) -> float:
    ref_tokens  = reference.strip().split()
    hyp_tokens  = hypothesis.strip().split()
    if len(hyp_tokens) < n:
        return 0.0
    ref_ngrams = Counter(_ngrams(ref_tokens, n))
    hyp_ngrams = Counter(_ngrams(hyp_tokens, n))
    clipped    = sum(min(cnt, ref_ngrams[g]) for g, cnt in hyp_ngrams.items())
    total      = sum(hyp_ngrams.values())
    return clipped / total if total else 0.0

N_EVAL = min(5000, len(df_full))
df_eval = df_full.sample(n=N_EVAL, random_state=0).reset_index(drop=True)
print(f'Evaluating on {len(df_eval):,} samples ...\n')

results = {
    'strict_exact': [],
    'any_valid_exact': [],
    'bleu1': [],
    'bleu2': [],
    'similarity': [],
    'confidence': [],
    'method': [],
}

for _, row in df_eval.iterrows():
    src  = row[SRC_COL]
    exp  = row[TGT_COL]
    res  = rag.lookup(src)
    pred = res.matched_tgt or ''

    strict = int(exp.strip() == pred.strip())
    all_tgts = res.all_targets or []
    any_valid = int(pred.strip() in [t.strip() for t in all_tgts]) if all_tgts else strict

    results['strict_exact'].append(strict)
    results['any_valid_exact'].append(any_valid)
    results['bleu1'].append(bleu_n(exp, pred, 1))
    results['bleu2'].append(bleu_n(exp, pred, 2))
    results['similarity'].append(res.similarity)
    results['confidence'].append(res.confidence)
    results['method'].append(res.method)

n_total    = len(df_eval)
n_strict   = sum(results['strict_exact'])
n_any      = sum(results['any_valid_exact'])
n_high     = results['confidence'].count('HIGH')
n_medium   = results['confidence'].count('MEDIUM')
n_low      = results['confidence'].count('LOW')
avg_bleu1  = sum(results['bleu1']) / n_total
avg_bleu2  = sum(results['bleu2']) / n_total
avg_sim    = sum(results['similarity']) / n_total
method_counts = Counter(results['method'])

print('=' * 60)
print('  EVALUATION RESULTS')
print('=' * 60)
print(f'  Total samples           : {n_total:,}')
print(f'  Strict exact match      : {n_strict:,} / {n_total:,}  ({100*n_strict/n_total:.1f}%)')
print(f'  Any-valid exact match   : {n_any:,} / {n_total:,}  ({100*n_any/n_total:.1f}%)')
print(f'  Avg BLEU-1              : {avg_bleu1:.4f}')
print(f'  Avg BLEU-2              : {avg_bleu2:.4f}')
print(f'  Avg similarity          : {avg_sim:.4f}')
print('-' * 60)
print(f'  Confidence:')
print(f'    HIGH   : {n_high:,}  ({100*n_high/n_total:.1f}%)')
print(f'    MEDIUM : {n_medium:,}  ({100*n_medium/n_total:.1f}%)')
print(f'    LOW    : {n_low:,}  ({100*n_low/n_total:.1f}%)')
print('-' * 60)
print(f'  Method:')
for method, count in method_counts.most_common():
    print(f'    {method:25s} : {count:,}  ({100*count/n_total:.1f}%)')
print('=' * 60)

non_valid = [i for i in range(n_total) if results['any_valid_exact'][i] == 0]
if non_valid:
    print(f'\n {len(non_valid)} entries where prediction is NOT among any valid targets:')
    for idx in non_valid[:5]:
        row = df_eval.iloc[idx]
        res = rag.lookup(row[SRC_COL])
        print(f'  src="{row[SRC_COL][:50]}"')
        print(f'    expected: "{row[TGT_COL][:60]}"')
        print(f'    got     : "{(res.matched_tgt or "")[:60]}"')
        print(f'    all_tgts: {res.all_targets}')
        print()
else:
    print('\n ALL predictions match a valid target! (100% any-valid exact match)')

Evaluating on 5,000 samples ...

  EVALUATION RESULTS
  Total samples           : 5,000
  Strict exact match      : 4,744 / 5,000  (94.9%)
  Any-valid exact match   : 5,000 / 5,000  (100.0%)
  Avg BLEU-1              : 0.9744
  Avg BLEU-2              : 0.9508
  Avg similarity          : 1.0000
------------------------------------------------------------
  Confidence:
    HIGH   : 5,000  (100.0%)
    MEDIUM : 0  (0.0%)
    LOW    : 0  (0.0%)
------------------------------------------------------------
  Method:
    exact_string              : 5,000  (100.0%)

 ALL predictions match a valid target! (100% any-valid exact match)


## 11. `translate()` API with Fallback

In [11]:
def dummy_transformer(query: str) -> str:
    return f'[Transformer would translate]: {query}'

tr1, info1 = rag.translate(sample_src, fallback_fn=dummy_transformer)
print(f'Query : {sample_src}')
print(f'Trans : {tr1}')
print(f'Info  : source={info1["source"]}, confidence={info1["confidence"]}, sim={info1["similarity"]:.4f}')
if info1.get('all_targets'):
    print(f'All valid: {info1["all_targets"][:3]}')
print()

tr2, info2 = rag.translate('xyz unknown nonsense', fallback_fn=dummy_transformer)
print(f'Query : xyz unknown nonsense')
print(f'Trans : {tr2}')
print(f'Info  : source={info2["source"]}, confidence={info2["confidence"]}, sim={info2["similarity"]:.4f}')

Query : wsrw nꞽt m n k ꞽrt ḥrw ꞽtḥtn f
Trans : Osiris Neith, nimm dir das Horusauge, das er herausgezogen hat.
Info  : source=RAG, confidence=HIGH, sim=1.0000
All valid: ['Osiris Neith, nimm dir das Horusauge, das er herausgezogen hat.']

Query : xyz unknown nonsense
Trans : [Transformer would translate]: xyz unknown nonsense
Info  : source=FALLBACK_MODEL, confidence=LOW, sim=0.2132


## 12. Save / Load

In [12]:
print(f'RAG saved at: {SAVE_PATH}')
print(f'Database size: {len(rag.df):,} entries')
print(f'Exact-string keys: {len(rag.exact_string_index):,}')
print(f'Word-set keys: {len(rag.exact_wordset_index):,}')
print(f'Char-sorted keys: {len(rag.char_sorted_index):,}')
print(f'TF-IDF vocab: {len(rag.tfidf.vocabulary_):,}')
print()
print('To load:')
print("  rag = HieroglyphRAG.load('/kaggle/working/rag_index.pkl')")

RAG saved at: /kaggle/working/rag_index.pkl
Database size: 89,562 entries
Exact-string keys: 84,908
Word-set keys: 84,130
Char-sorted keys: 84,458
TF-IDF vocab: 21,346

To load:
  rag = HieroglyphRAG.load('/kaggle/working/rag_index.pkl')


In [13]:
tr1, info1 = rag.translate("jšst pw jr f rḥw", fallback_fn=dummy_transformer)
print(f'Query : jšst pw jr f rḥw')
print(f'Trans : {tr1}')
print(f'Info  : source={info1["source"]}, confidence={info1["confidence"]}, sim={info1["similarity"]:.4f}')

Query : jšst pw jr f rḥw
Trans : Was ist das denn, Leute?
Info  : source=RAG, confidence=HIGH, sim=1.0000


In [14]:
tr1, info1 = rag.translate("jšt pw jr f rḥ", fallback_fn=dummy_transformer)
print(f'Query : jšt pw jr f rḥ')
print(f'Trans : {tr1}')
print(f'Info  : source={info1["source"]}, confidence={info1["confidence"]}, sim={info1["similarity"]:.4f}')

Query : jšt pw jr f rḥ
Trans : [Transformer would translate]: jšt pw jr f rḥ
Info  : source=FALLBACK_MODEL, confidence=LOW, sim=0.6401
